In [ ]:
# ============================================================
# Pixel-space deterministic residual CNN
# ERA5 Tmax controlled downscaling experiment
# ============================================================

from pathlib import Path
import random

import numpy as np
import xarray as xr
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# CPU configuration
DEVICE = torch.device("cpu")

print("Device:", DEVICE)

In [ ]:
## Data

#This notebook expects normalized ERA5 Tmax files in a local `data/`directory. Raw and processed NetCDF files are not included in the GitHub repository.

#The chronological split is:

#Training: 1950–2005
#Validation: 2006–2014
#Testing: 2015–2024



# ============================================================
# Project paths
# ============================================================

PROJECT_DIR = Path("..")

DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)


TRAIN_FILE = DATA_DIR / "ERA5_Tmax_train_normalized.nc"
VAL_FILE = DATA_DIR / "ERA5_Tmax_validation_normalized.nc"
TEST_FILE = DATA_DIR / "ERA5_Tmax_test_normalized.nc"

CHECKPOINT_FILE = CHECKPOINT_DIR / "pixel_cnn.pt"
TRAIN_LOSS_FILE = RESULTS_DIR / "pixel_cnn_train_losses.npy"
VAL_LOSS_FILE = RESULTS_DIR / "pixel_cnn_val_losses.npy"

In [ ]:
train_ds = xr.open_dataset(TRAIN_FILE)
val_ds = xr.open_dataset(VAL_FILE)
test_ds = xr.open_dataset(TEST_FILE)

train_da = train_ds["tmax"]
val_da = val_ds["tmax"]
test_da = test_ds["tmax"]

print("Training:", train_da.shape)
print("Validation:", val_da.shape)
print("Testing:", test_da.shape)

In [ ]:
class ERA5DownscalingDataset(Dataset):
    """
    Construct paired coarse- and high-resolution ERA5 temperature fields.

    The normalized 0.25-degree ERA5 field is treated as the target.
    It is coarsened by a factor of four using area averaging and then
    bilinearly interpolated back to the original grid.

    The model learns the residual between the ERA5 target and this
    bilinear baseline.
    """

    def __init__(self, data_array, coarse_factor=4):
        self.data = data_array
        self.coarse_factor = coarse_factor

    def __len__(self):
        return self.data.sizes["valid_time"]

    def __getitem__(self, idx):
        field = (
            self.data
            .isel(valid_time=idx)
            .values
            .astype(np.float32)
        )

        target = torch.from_numpy(field).unsqueeze(0)

        x = target.unsqueeze(0)

        # 48 x 60 -> 12 x 15
        coarse_small = F.interpolate(
            x,
            scale_factor=1 / self.coarse_factor,
            mode="area",
        )

        # 12 x 15 -> 48 x 60
        bilinear = F.interpolate(
            coarse_small,
            size=target.shape[-2:],
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)

        return {
            "target": target,
            "bilinear": bilinear,
        }

In [ ]:
full_train_dataset = ERA5DownscalingDataset(train_da)
full_val_dataset = ERA5DownscalingDataset(val_da)
test_dataset = ERA5DownscalingDataset(test_da)


# CPU-friendly proof-of-concept subsets
rng = np.random.default_rng(SEED)

train_indices = rng.choice(
    len(full_train_dataset),
    size=8000,
    replace=False,
)

val_indices = rng.choice(
    len(full_val_dataset),
    size=1500,
    replace=False,
)

train_dataset = Subset(full_train_dataset, train_indices)
val_dataset = Subset(full_val_dataset, val_indices)


BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

In [ ]:
class ResidualCNN(nn.Module):
    """Deterministic CNN that predicts the missing fine-scale residual."""

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),

            nn.Conv2d(16, 16, 3, padding=1),
            nn.ReLU(),

            nn.Conv2d(16, 16, 3, padding=1),
            nn.ReLU(),

            nn.Conv2d(16, 8, 3, padding=1),
            nn.ReLU(),

            nn.Conv2d(8, 1, 3, padding=1),
        )

    def forward(self, x):
        return self.net(x)


model = ResidualCNN().to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters: {n_params:,}")

In [ ]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

N_EPOCHS = 5

train_losses = []
val_losses = []

best_val_loss = np.inf


for epoch in range(N_EPOCHS):

    # -------------------------
    # Training
    # -------------------------
    model.train()

    running_loss = 0.0
    n_samples = 0

    for batch in train_loader:

        bilinear = batch["bilinear"].to(DEVICE)
        target = batch["target"].to(DEVICE)

        true_residual = target - bilinear

        pred_residual = model(bilinear)

        loss = criterion(
            pred_residual,
            true_residual,
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = bilinear.size(0)

        running_loss += loss.item() * batch_size
        n_samples += batch_size

    train_loss = running_loss / n_samples


    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    running_loss = 0.0
    n_samples = 0

    with torch.no_grad():

        for batch in val_loader:

            bilinear = batch["bilinear"].to(DEVICE)
            target = batch["target"].to(DEVICE)

            true_residual = target - bilinear
            pred_residual = model(bilinear)

            loss = criterion(
                pred_residual,
                true_residual,
            )

            batch_size = bilinear.size(0)

            running_loss += loss.item() * batch_size
            n_samples += batch_size

    val_loss = running_loss / n_samples

    train_losses.append(train_loss)
    val_losses.append(val_loss)


    # Save best validation model
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
            },
            CHECKPOINT_FILE,
        )

    print(
        f"Epoch {epoch + 1:02d}/{N_EPOCHS} | "
        f"Train: {train_loss:.6f} | "
        f"Validation: {val_loss:.6f}"
    )


np.save(TRAIN_LOSS_FILE, np.asarray(train_losses))
np.save(VAL_LOSS_FILE, np.asarray(val_losses))

print("\nBest validation loss:", best_val_loss)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

epochs = np.arange(1, len(train_losses) + 1)

ax.plot(
    epochs,
    train_losses,
    marker="o",
    label="Training",
)

ax.plot(
    epochs,
    val_losses,
    marker="o",
    label="Validation",
)

ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.legend()

fig.tight_layout()

figure_file = FIGURES_DIR / "pixel_cnn_training_loss.png"

fig.savefig(
    figure_file,
    dpi=200,
    bbox_inches="tight",
)

plt.close(fig)

print("Saved:", figure_file)

In [ ]:
checkpoint = torch.load(
    CHECKPOINT_FILE,
    map_location=DEVICE,
)

model.load_state_dict(checkpoint["model_state"])
model.eval()


bilinear_squared_error = 0.0
cnn_squared_error = 0.0
n_values = 0


with torch.no_grad():

    for batch in test_loader:

        bilinear = batch["bilinear"].to(DEVICE)
        target = batch["target"].to(DEVICE)

        pred_residual = model(bilinear)

        prediction = bilinear + pred_residual

        bilinear_squared_error += (
            (bilinear - target) ** 2
        ).sum().item()

        cnn_squared_error += (
            (prediction - target) ** 2
        ).sum().item()

        n_values += target.numel()


bilinear_rmse = np.sqrt(
    bilinear_squared_error / n_values
)

cnn_rmse = np.sqrt(
    cnn_squared_error / n_values
)

improvement = (
    (bilinear_rmse - cnn_rmse)
    / bilinear_rmse
    * 100
)


print("Held-out test period: 2015–2024")
print(f"Bilinear RMSE: {bilinear_rmse:.5f}")
print(f"Pixel CNN RMSE: {cnn_rmse:.5f}")
print(f"Improvement: {improvement:.2f}%")

In [ ]:
baseline_results = pd.DataFrame(
    {
        "method": [
            "Bilinear interpolation",
            "Pixel CNN",
        ],
        "rmse": [
            bilinear_rmse,
            cnn_rmse,
        ],
    }
)

results_file = RESULTS_DIR / "pixel_cnn_test_results.csv"

baseline_results.to_csv(
    results_file,
    index=False,
)

baseline_results